In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
position:
  x: 570
  y: 304
description:
  text: Load all data from the specified nutrition and obesity table.
  hash: 4ac65bb3
previewCodeHash: b3cb1160179d5d33
previewMode: full
lastPreviewRowLimit: "1168"
config:
  table_source:
    tableName: nutrition_and_obesity_data.`default`.nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "nutrition_and_obesity_data.`default`.nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]
if globals().get("ld_display_outputs", True):
    display(ctx["source_0.data"])

In [0]:
"""
id: dataset_shape
template: sql
templateVersion: 1.0.0
name: dataset_shape
position:
  x: 830
  y: 164
description:
  text: Count total rows and report total columns as 33 in the data.
  hash: a5efc8e1
previewCodeHash: dfba5325489f362d
previewMode: full
config:
  query: |
    SELECT
      COUNT(*) AS total_rows,
      33 AS total_columns
    FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT\n  COUNT(*) AS total_rows,\n  33 AS total_columns\nFROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n"
}
inputs = {
    "data": [
        ctx["source_0.data"]
    ],
    "data__sources": [
        {
            "node": "source_0",
            "output_port": "data",
            "name": "nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system",
            "df_name": "nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system"
        }
    ]
}
out = run(config, inputs, spark)
ctx["dataset_shape.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["dataset_shape.result"])

In [0]:
"""
id: drop_useless_columns
template: transform
templateVersion: 2.0.0
name: drop_useless_columns
position:
  x: 830
  y: 500
description:
  text: Remove specified columns from the data.
  hash: 7859ad93
previewCodeHash: 58baf4e640174c34
previewMode: "1000"
config:
  mode: passthrough
  edits:
    - column: Data_Value_Unit
      checked: false
    - column: Total
      checked: false
    - column: Data_Value_Footnote_Symbol
      checked: false
    - column: Data_Value_Footnote
      checked: false
    - column: Data_Value_Alt
      checked: false
    - column: LocationID
      checked: false
    - column: LocationAbbr
      checked: false
    - column: ClassID
      checked: false
    - column: TopicID
      checked: false
    - column: QuestionID
      checked: false
    - column: DataValueTypeID
      checked: false
    - column: StratificationCategoryId1
      checked: false
    - column: StratificationID1
      checked: false
  ordered: []
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Dict, List
from pyspark.sql import functions as F

def _col_ref(name: str):
    if "." in name and not (name.startswith("`") and name.endswith("`")):
        return F.col("`" + name.replace("`", "``") + "`")
    return F.col(name)

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _column_for(item: Dict[str, Any]):
    col = _col_ref(item.get("column", ""))
    alias = item.get("alias")
    if alias:
        col = col.alias(alias)
    return col

def _passthrough_column(name: str, rename_map: Dict[str, Dict[str, Any]]):
    entry = rename_map.get(name)
    col = _col_ref(name)
    if entry and entry.get("alias"):
        col = col.alias(entry["alias"])
    return col

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    mode = config.get("mode", "passthrough")
    edits: List[Dict[str, Any]] = config.get("edits", [])
    ordered: List[str] = config.get("ordered") or []

    if mode == "select":
        checked_edits = [item for item in edits if _is_checked(item)]
        if not checked_edits:
            return {"transformed_data": df}
        order_index = {name: i for i, name in enumerate(ordered)}
        tail = len(order_index)
        ordered_edits = sorted(
            checked_edits,
            key=lambda item: order_index.get(item.get("column", ""), tail),
        )
        return {"transformed_data": df.select(*(_column_for(item) for item in ordered_edits))}

    unchecked_cols = {
        item.get("column", "")
        for item in edits
        if not _is_checked(item)
    }
    rename_map = {
        item.get("column", ""): item
        for item in edits
        if item.get("alias") and _is_checked(item)
    }
    upstream_cols = list(df.columns)
    upstream_set = set(upstream_cols)

    effective_ordered = ordered if ordered else list(upstream_cols)

    placed = set()
    out = []
    for token in effective_ordered:
        if token in placed or token not in upstream_set:
            continue
        placed.add(token)
        if token in unchecked_cols:
            continue
        out.append(_passthrough_column(token, rename_map))

    for col in upstream_cols:
        if col in placed or col in unchecked_cols:
            continue
        out.append(_passthrough_column(col, rename_map))

    if not out:
        return {"transformed_data": df}
    return {"transformed_data": df.select(*out)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "mode": "passthrough",
    "edits": [
        {
            "column": "Data_Value_Unit",
            "checked": False
        },
        {
            "column": "Total",
            "checked": False
        },
        {
            "column": "Data_Value_Footnote_Symbol",
            "checked": False
        },
        {
            "column": "Data_Value_Footnote",
            "checked": False
        },
        {
            "column": "Data_Value_Alt",
            "checked": False
        },
        {
            "column": "LocationID",
            "checked": False
        },
        {
            "column": "LocationAbbr",
            "checked": False
        },
        {
            "column": "ClassID",
            "checked": False
        },
        {
            "column": "TopicID",
            "checked": False
        },
        {
            "column": "QuestionID",
            "checked": False
        },
        {
            "column": "DataValueTypeID",
            "checked": False
        },
        {
            "column": "StratificationCategoryId1",
            "checked": False
        },
        {
            "column": "StratificationID1",
            "checked": False
        }
    ],
    "ordered": []
}
inputs = {
    "data": ctx["source_0.data"]
}
out = run(config, inputs, spark)
ctx["drop_useless_columns.transformed_data"] = out["transformed_data"]
if globals().get("ld_display_outputs", True):
    display(ctx["drop_useless_columns.transformed_data"])

In [0]:
"""
id: footnote_check
template: sql
templateVersion: 1.0.0
name: footnote_check
position:
  x: 1350
  y: 444
description:
  text: Count rows with missing Data_Value grouped by footnote.
  hash: 725814f6
previewCodeHash: e57d1cb393b73254
previewMode: "1000"
config:
  query: |
    SELECT
      Data_Value_Footnote,
      COUNT(*) AS row_count
    FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
    WHERE Data_Value IS NULL
    GROUP BY Data_Value_Footnote
    ORDER BY row_count DESC
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT\n  Data_Value_Footnote,\n  COUNT(*) AS row_count\nFROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\nWHERE Data_Value IS NULL\nGROUP BY Data_Value_Footnote\nORDER BY row_count DESC\n"
}
inputs = {
    "data": [
        ctx["source_0.data"]
    ],
    "data__sources": [
        {
            "node": "source_0",
            "output_port": "data",
            "name": "nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system",
            "df_name": "nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system"
        }
    ]
}
out = run(config, inputs, spark)
ctx["footnote_check.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["footnote_check.result"])

In [0]:
"""
id: full_null_analysis
template: sql
templateVersion: 1.0.0
name: full_null_analysis
position:
  x: 1090
  y: 304
description:
  text: Calculate and show the percentage of missing and non-missing values for each column.
  hash: 3b6a0a31
previewCodeHash: 4e3533ccfe98222c
previewMode: full
config:
  query: |
    WITH total AS (
      SELECT COUNT(*) AS total_rows
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
    )
    SELECT column_name, null_count,
           ROUND(null_count * 100.0 / total_rows, 2) AS null_percentage,
           ROUND((total_rows - null_count) * 100.0 / total_rows, 2) AS filled_percentage
    FROM total
    CROSS JOIN (
      SELECT 'YearStart' AS column_name, SUM(CASE WHEN YearStart IS NULL THEN 1 ELSE 0 END) AS null_count FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'YearEnd', SUM(CASE WHEN YearEnd IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'LocationAbbr', SUM(CASE WHEN LocationAbbr IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'LocationDesc', SUM(CASE WHEN LocationDesc IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Datasource', SUM(CASE WHEN Datasource IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Class', SUM(CASE WHEN Class IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Topic', SUM(CASE WHEN Topic IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Question', SUM(CASE WHEN Question IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Data_Value_Unit', SUM(CASE WHEN Data_Value_Unit IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Data_Value_Type', SUM(CASE WHEN Data_Value_Type IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Data_Value', SUM(CASE WHEN Data_Value IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Data_Value_Alt', SUM(CASE WHEN Data_Value_Alt IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Data_Value_Footnote_Symbol', SUM(CASE WHEN Data_Value_Footnote_Symbol IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Data_Value_Footnote', SUM(CASE WHEN Data_Value_Footnote IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Low_Confidence_Limit', SUM(CASE WHEN Low_Confidence_Limit IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'High_Confidence_Limit', SUM(CASE WHEN `High_Confidence_Limit ` IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Sample_Size', SUM(CASE WHEN Sample_Size IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Total', SUM(CASE WHEN Total IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Age(years)', SUM(CASE WHEN `Age(years)` IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Education', SUM(CASE WHEN Education IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Sex', SUM(CASE WHEN Sex IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Income', SUM(CASE WHEN Income IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Race/Ethnicity', SUM(CASE WHEN `Race/Ethnicity` IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'GeoLocation', SUM(CASE WHEN GeoLocation IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'ClassID', SUM(CASE WHEN ClassID IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'TopicID', SUM(CASE WHEN TopicID IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'QuestionID', SUM(CASE WHEN QuestionID IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'DataValueTypeID', SUM(CASE WHEN DataValueTypeID IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'LocationID', SUM(CASE WHEN LocationID IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'StratificationCategory1', SUM(CASE WHEN StratificationCategory1 IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'Stratification1', SUM(CASE WHEN Stratification1 IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'StratificationCategoryId1', SUM(CASE WHEN StratificationCategoryId1 IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL SELECT 'StratificationID1', SUM(CASE WHEN StratificationID1 IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
    ) nulls
    ORDER BY null_percentage DESC
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "WITH total AS (\n  SELECT COUNT(*) AS total_rows\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n)\nSELECT column_name, null_count,\n       ROUND(null_count * 100.0 / total_rows, 2) AS null_percentage,\n       ROUND((total_rows - null_count) * 100.0 / total_rows, 2) AS filled_percentage\nFROM total\nCROSS JOIN (\n  SELECT 'YearStart' AS column_name, SUM(CASE WHEN YearStart IS NULL THEN 1 ELSE 0 END) AS null_count FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'YearEnd', SUM(CASE WHEN YearEnd IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'LocationAbbr', SUM(CASE WHEN LocationAbbr IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'LocationDesc', SUM(CASE WHEN LocationDesc IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Datasource', SUM(CASE WHEN Datasource IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Class', SUM(CASE WHEN Class IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Topic', SUM(CASE WHEN Topic IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Question', SUM(CASE WHEN Question IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Data_Value_Unit', SUM(CASE WHEN Data_Value_Unit IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Data_Value_Type', SUM(CASE WHEN Data_Value_Type IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Data_Value', SUM(CASE WHEN Data_Value IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Data_Value_Alt', SUM(CASE WHEN Data_Value_Alt IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Data_Value_Footnote_Symbol', SUM(CASE WHEN Data_Value_Footnote_Symbol IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Data_Value_Footnote', SUM(CASE WHEN Data_Value_Footnote IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Low_Confidence_Limit', SUM(CASE WHEN Low_Confidence_Limit IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'High_Confidence_Limit', SUM(CASE WHEN `High_Confidence_Limit ` IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Sample_Size', SUM(CASE WHEN Sample_Size IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Total', SUM(CASE WHEN Total IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Age(years)', SUM(CASE WHEN `Age(years)` IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Education', SUM(CASE WHEN Education IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Sex', SUM(CASE WHEN Sex IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Income', SUM(CASE WHEN Income IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Race/Ethnicity', SUM(CASE WHEN `Race/Ethnicity` IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'GeoLocation', SUM(CASE WHEN GeoLocation IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'ClassID', SUM(CASE WHEN ClassID IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'TopicID', SUM(CASE WHEN TopicID IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'QuestionID', SUM(CASE WHEN QuestionID IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'DataValueTypeID', SUM(CASE WHEN DataValueTypeID IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'LocationID', SUM(CASE WHEN LocationID IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'StratificationCategory1', SUM(CASE WHEN StratificationCategory1 IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'Stratification1', SUM(CASE WHEN Stratification1 IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'StratificationCategoryId1', SUM(CASE WHEN StratificationCategoryId1 IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL SELECT 'StratificationID1', SUM(CASE WHEN StratificationID1 IS NULL THEN 1 ELSE 0 END) FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n) nulls\nORDER BY null_percentage DESC\n"
}
inputs = {
    "data": [
        ctx["source_0.data"]
    ],
    "data__sources": [
        {
            "node": "source_0",
            "output_port": "data",
            "name": "nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system",
            "df_name": "nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system"
        }
    ]
}
out = run(config, inputs, spark)
ctx["full_null_analysis.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["full_null_analysis.result"])

In [0]:
"""
id: location_id_check
template: sql
templateVersion: 1.0.0
name: location_id_check
position:
  x: 1350
  y: 304
description:
  text: Count unique location IDs, names, and abbreviations.
  hash: 34e9229a
previewCodeHash: 1252f9647f12bb77
previewMode: "1000"
config:
  query: |
    SELECT
      COUNT(DISTINCT LocationID) AS distinct_location_ids,
      COUNT(DISTINCT LocationDesc) AS distinct_location_names,
      COUNT(DISTINCT LocationAbbr) AS distinct_location_abbrs
    FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT\n  COUNT(DISTINCT LocationID) AS distinct_location_ids,\n  COUNT(DISTINCT LocationDesc) AS distinct_location_names,\n  COUNT(DISTINCT LocationAbbr) AS distinct_location_abbrs\nFROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n"
}
inputs = {
    "data": [
        ctx["source_0.data"]
    ],
    "data__sources": [
        {
            "node": "source_0",
            "output_port": "data",
            "name": "nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system",
            "df_name": "nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system"
        }
    ]
}
out = run(config, inputs, spark)
ctx["location_id_check.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["location_id_check.result"])

In [0]:
"""
id: null_analysis
template: sql
templateVersion: 1.0.0
name: null_analysis
position:
  x: 830
  y: 304
description:
  text: Calculate and show the count and percentage of missing data for each column.
  hash: 0b90dc1b
previewCodeHash: 46142333040cece6
previewMode: full
config:
  query: |
    WITH total AS (
      SELECT COUNT(*) AS total_rows
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
    )
    SELECT column_name, null_count,
           ROUND(null_count * 100.0 / total_rows, 2) AS null_percentage
    FROM total
    CROSS JOIN (
      SELECT 'YearStart' AS column_name, SUM(CASE WHEN YearStart IS NULL THEN 1 ELSE 0 END) AS null_count
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'YearEnd', SUM(CASE WHEN YearEnd IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'LocationAbbr', SUM(CASE WHEN LocationAbbr IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'LocationDesc', SUM(CASE WHEN LocationDesc IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Datasource', SUM(CASE WHEN Datasource IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Class', SUM(CASE WHEN Class IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Topic', SUM(CASE WHEN Topic IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Question', SUM(CASE WHEN Question IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Data_Value_Unit', SUM(CASE WHEN Data_Value_Unit IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Data_Value_Type', SUM(CASE WHEN Data_Value_Type IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Data_Value', SUM(CASE WHEN Data_Value IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Data_Value_Alt', SUM(CASE WHEN Data_Value_Alt IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Data_Value_Footnote_Symbol', SUM(CASE WHEN Data_Value_Footnote_Symbol IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Data_Value_Footnote', SUM(CASE WHEN Data_Value_Footnote IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Low_Confidence_Limit', SUM(CASE WHEN Low_Confidence_Limit IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'High_Confidence_Limit', SUM(CASE WHEN `High_Confidence_Limit ` IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Sample_Size', SUM(CASE WHEN Sample_Size IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Total', SUM(CASE WHEN Total IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Age(years)', SUM(CASE WHEN `Age(years)` IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Education', SUM(CASE WHEN Education IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Sex', SUM(CASE WHEN Sex IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Income', SUM(CASE WHEN Income IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Race/Ethnicity', SUM(CASE WHEN `Race/Ethnicity` IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'GeoLocation', SUM(CASE WHEN GeoLocation IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'ClassID', SUM(CASE WHEN ClassID IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'TopicID', SUM(CASE WHEN TopicID IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'QuestionID', SUM(CASE WHEN QuestionID IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'DataValueTypeID', SUM(CASE WHEN DataValueTypeID IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'LocationID', SUM(CASE WHEN LocationID IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'StratificationCategory1', SUM(CASE WHEN StratificationCategory1 IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'Stratification1', SUM(CASE WHEN Stratification1 IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'StratificationCategoryId1', SUM(CASE WHEN StratificationCategoryId1 IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
      UNION ALL
      SELECT 'StratificationID1', SUM(CASE WHEN StratificationID1 IS NULL THEN 1 ELSE 0 END)
      FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
    ) nulls
    ORDER BY null_percentage DESC
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "WITH total AS (\n  SELECT COUNT(*) AS total_rows\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n)\nSELECT column_name, null_count,\n       ROUND(null_count * 100.0 / total_rows, 2) AS null_percentage\nFROM total\nCROSS JOIN (\n  SELECT 'YearStart' AS column_name, SUM(CASE WHEN YearStart IS NULL THEN 1 ELSE 0 END) AS null_count\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'YearEnd', SUM(CASE WHEN YearEnd IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'LocationAbbr', SUM(CASE WHEN LocationAbbr IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'LocationDesc', SUM(CASE WHEN LocationDesc IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Datasource', SUM(CASE WHEN Datasource IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Class', SUM(CASE WHEN Class IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Topic', SUM(CASE WHEN Topic IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Question', SUM(CASE WHEN Question IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Data_Value_Unit', SUM(CASE WHEN Data_Value_Unit IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Data_Value_Type', SUM(CASE WHEN Data_Value_Type IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Data_Value', SUM(CASE WHEN Data_Value IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Data_Value_Alt', SUM(CASE WHEN Data_Value_Alt IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Data_Value_Footnote_Symbol', SUM(CASE WHEN Data_Value_Footnote_Symbol IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Data_Value_Footnote', SUM(CASE WHEN Data_Value_Footnote IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Low_Confidence_Limit', SUM(CASE WHEN Low_Confidence_Limit IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'High_Confidence_Limit', SUM(CASE WHEN `High_Confidence_Limit ` IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Sample_Size', SUM(CASE WHEN Sample_Size IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Total', SUM(CASE WHEN Total IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Age(years)', SUM(CASE WHEN `Age(years)` IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Education', SUM(CASE WHEN Education IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Sex', SUM(CASE WHEN Sex IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Income', SUM(CASE WHEN Income IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Race/Ethnicity', SUM(CASE WHEN `Race/Ethnicity` IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'GeoLocation', SUM(CASE WHEN GeoLocation IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'ClassID', SUM(CASE WHEN ClassID IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'TopicID', SUM(CASE WHEN TopicID IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'QuestionID', SUM(CASE WHEN QuestionID IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'DataValueTypeID', SUM(CASE WHEN DataValueTypeID IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'LocationID', SUM(CASE WHEN LocationID IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'StratificationCategory1', SUM(CASE WHEN StratificationCategory1 IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'Stratification1', SUM(CASE WHEN Stratification1 IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'StratificationCategoryId1', SUM(CASE WHEN StratificationCategoryId1 IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n  UNION ALL\n  SELECT 'StratificationID1', SUM(CASE WHEN StratificationID1 IS NULL THEN 1 ELSE 0 END)\n  FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\n) nulls\nORDER BY null_percentage DESC\n"
}
inputs = {
    "data": [
        ctx["source_0.data"]
    ],
    "data__sources": [
        {
            "node": "source_0",
            "output_port": "data",
            "name": "nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system",
            "df_name": "nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system"
        }
    ]
}
out = run(config, inputs, spark)
ctx["null_analysis.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["null_analysis.result"])

In [0]:
"""
id: stratification_values
template: sql
templateVersion: 1.0.0
name: stratification_values
position:
  x: 1350
  y: 164
description:
  text: Group by category, count distinct values, and show sample values.
  hash: 41ccd7a4
previewCodeHash: 35ca1a9df70a3a88
previewMode: "1000"
config:
  query: |
    SELECT
      StratificationCategory1,
      COUNT(DISTINCT Stratification1) AS distinct_values,
      CONCAT_WS(', ', COLLECT_SET(Stratification1)) AS sample_values
    FROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system
    GROUP BY StratificationCategory1
    ORDER BY StratificationCategory1
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT\n  StratificationCategory1,\n  COUNT(DISTINCT Stratification1) AS distinct_values,\n  CONCAT_WS(', ', COLLECT_SET(Stratification1)) AS sample_values\nFROM nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system\nGROUP BY StratificationCategory1\nORDER BY StratificationCategory1\n"
}
inputs = {
    "data": [
        ctx["source_0.data"]
    ],
    "data__sources": [
        {
            "node": "source_0",
            "output_port": "data",
            "name": "nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system",
            "df_name": "nutrition_physical_activity_and_obesity_behavioral_risk_factor_surveillance_system"
        }
    ]
}
out = run(config, inputs, spark)
ctx["stratification_values.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["stratification_values.result"])

In [0]:
"""
id: filter_null_data_values
template: filter
templateVersion: 2.0.0
name: clean_base_dataset
position:
  x: 1090
  y: 500
description:
  text: Keep rows where Data_Value is not empty and separate the rest.
  hash: d29f0c72
previewCodeHash: dbcda7f8c7020c40
previewMode: full
config:
  condition: Data_Value IS NOT NULL AND Stratification1 != 'Data not reported'
input:
  - node: drop_useless_columns
    input_port: data
    output_port: transformed_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "Data_Value IS NOT NULL AND Stratification1 != 'Data not reported'"
}
inputs = {
    "data": ctx["drop_useless_columns.transformed_data"]
}
out = run(config, inputs, spark)
ctx["filter_null_data_values.filtered_data"] = out["filtered_data"]
ctx["filter_null_data_values.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", True):
    display(ctx["filter_null_data_values.filtered_data"])
    display(ctx["filter_null_data_values.excluded_data"])

In [0]:
"""
id: correlation_analysis
template: sql
templateVersion: 1.0.0
name: Correlation Analysis - Factors vs Obesity
position:
  x: 1640
  y: 1418
description:
  text: Analyze which factors are most correlated with obesity by state and year.
  hash: c73eb6e0
previewCodeHash: eeaefadda3560b2f
previewMode: full
config:
  query: |
    -- Correlation analysis: which factors are most correlated with obesity?
    -- Approach: At the state-year level, correlate obesity rate with each factor

    WITH obesity_by_state_year AS (
      SELECT LocationDesc, YearStart,
             ROUND(AVG(Data_Value), 2) AS obesity_rate
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Total'
      GROUP BY LocationDesc, YearStart
    ),

    -- Physical inactivity rate per state-year (Total)
    inactivity_by_state_year AS (
      SELECT LocationDesc, YearStart,
             ROUND(AVG(Data_Value), 2) AS inactivity_rate
      FROM clean_base_dataset
      WHERE Class = 'Physical Activity'
        AND StratificationCategory1 = 'Total'
      GROUP BY LocationDesc, YearStart
    ),

    -- Poor nutrition rate per state-year (Total)
    nutrition_by_state_year AS (
      SELECT LocationDesc, YearStart,
             ROUND(AVG(Data_Value), 2) AS poor_nutrition_rate
      FROM clean_base_dataset
      WHERE Class = 'Fruits and Vegetables'
        AND StratificationCategory1 = 'Total'
      GROUP BY LocationDesc, YearStart
    ),

    -- Income encoding: higher income bracket = higher number
    income_encoded AS (
      SELECT LocationDesc, YearStart,
             AVG(CASE Stratification1
               WHEN 'Less than $15,000' THEN 1
               WHEN '$15,000 - $24,999' THEN 2
               WHEN '$25,000 - $34,999' THEN 3
               WHEN '$35,000 - $49,999' THEN 4
               WHEN '$50,000 - $74,999' THEN 5
               WHEN '$75,000 or greater' THEN 6
               ELSE NULL END * Data_Value) / NULLIF(AVG(Data_Value), 0) AS weighted_income_index
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Income'
      GROUP BY LocationDesc, YearStart
    ),

    -- Education encoding: higher education = higher number
    education_encoded AS (
      SELECT LocationDesc, YearStart,
             AVG(CASE Stratification1
               WHEN 'Less than high school' THEN 1
               WHEN 'High school graduate' THEN 2
               WHEN 'Some college or technical school' THEN 3
               WHEN 'College graduate' THEN 4
               ELSE NULL END * Data_Value) / NULLIF(AVG(Data_Value), 0) AS weighted_education_index
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Education'
      GROUP BY LocationDesc, YearStart
    ),

    -- Age encoding: midpoint of each age bracket
    age_encoded AS (
      SELECT LocationDesc, YearStart,
             AVG(CASE Stratification1
               WHEN '18 - 24' THEN 21
               WHEN '25 - 34' THEN 30
               WHEN '35 - 44' THEN 40
               WHEN '45 - 54' THEN 50
               WHEN '55 - 64' THEN 60
               WHEN '65 or older' THEN 70
               ELSE NULL END * Data_Value) / NULLIF(AVG(Data_Value), 0) AS weighted_age_index
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Age (years)'
      GROUP BY LocationDesc, YearStart
    )

    SELECT
      'Physical Inactivity' AS factor,
      ROUND(CORR(o.obesity_rate, i.inactivity_rate), 4) AS correlation_coefficient,
      COUNT(*) AS data_points,
      CASE
        WHEN ABS(CORR(o.obesity_rate, i.inactivity_rate)) >= 0.7 THEN 'Strong'
        WHEN ABS(CORR(o.obesity_rate, i.inactivity_rate)) >= 0.4 THEN 'Moderate'
        ELSE 'Weak'
      END AS strength
    FROM obesity_by_state_year o
    JOIN inactivity_by_state_year i ON o.LocationDesc = i.LocationDesc AND o.YearStart = i.YearStart

    UNION ALL

    SELECT
      'Poor Nutrition (Low Fruit/Veg)' AS factor,
      ROUND(CORR(o.obesity_rate, n.poor_nutrition_rate), 4) AS correlation_coefficient,
      COUNT(*) AS data_points,
      CASE
        WHEN ABS(CORR(o.obesity_rate, n.poor_nutrition_rate)) >= 0.7 THEN 'Strong'
        WHEN ABS(CORR(o.obesity_rate, n.poor_nutrition_rate)) >= 0.4 THEN 'Moderate'
        ELSE 'Weak'
      END AS strength
    FROM obesity_by_state_year o
    JOIN nutrition_by_state_year n ON o.LocationDesc = n.LocationDesc AND o.YearStart = n.YearStart

    UNION ALL

    SELECT
      'Lower Income (weighted index)' AS factor,
      ROUND(CORR(o.obesity_rate, inc.weighted_income_index), 4) AS correlation_coefficient,
      COUNT(*) AS data_points,
      CASE
        WHEN ABS(CORR(o.obesity_rate, inc.weighted_income_index)) >= 0.7 THEN 'Strong'
        WHEN ABS(CORR(o.obesity_rate, inc.weighted_income_index)) >= 0.4 THEN 'Moderate'
        ELSE 'Weak'
      END AS strength
    FROM obesity_by_state_year o
    JOIN income_encoded inc ON o.LocationDesc = inc.LocationDesc AND o.YearStart = inc.YearStart

    UNION ALL

    SELECT
      'Lower Education (weighted index)' AS factor,
      ROUND(CORR(o.obesity_rate, edu.weighted_education_index), 4) AS correlation_coefficient,
      COUNT(*) AS data_points,
      CASE
        WHEN ABS(CORR(o.obesity_rate, edu.weighted_education_index)) >= 0.7 THEN 'Strong'
        WHEN ABS(CORR(o.obesity_rate, edu.weighted_education_index)) >= 0.4 THEN 'Moderate'
        ELSE 'Weak'
      END AS strength
    FROM obesity_by_state_year o
    JOIN education_encoded edu ON o.LocationDesc = edu.LocationDesc AND o.YearStart = edu.YearStart

    UNION ALL

    SELECT
      'Older Age (weighted index)' AS factor,
      ROUND(CORR(o.obesity_rate, age.weighted_age_index), 4) AS correlation_coefficient,
      COUNT(*) AS data_points,
      CASE
        WHEN ABS(CORR(o.obesity_rate, age.weighted_age_index)) >= 0.7 THEN 'Strong'
        WHEN ABS(CORR(o.obesity_rate, age.weighted_age_index)) >= 0.4 THEN 'Moderate'
        ELSE 'Weak'
      END AS strength
    FROM obesity_by_state_year o
    JOIN age_encoded age ON o.LocationDesc = age.LocationDesc AND o.YearStart = age.YearStart

    ORDER BY ABS(correlation_coefficient) DESC
input:
  - node: filter_null_data_values
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "-- Correlation analysis: which factors are most correlated with obesity?\n-- Approach: At the state-year level, correlate obesity rate with each factor\n\nWITH obesity_by_state_year AS (\n  SELECT LocationDesc, YearStart,\n         ROUND(AVG(Data_Value), 2) AS obesity_rate\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Total'\n  GROUP BY LocationDesc, YearStart\n),\n\n-- Physical inactivity rate per state-year (Total)\ninactivity_by_state_year AS (\n  SELECT LocationDesc, YearStart,\n         ROUND(AVG(Data_Value), 2) AS inactivity_rate\n  FROM clean_base_dataset\n  WHERE Class = 'Physical Activity'\n    AND StratificationCategory1 = 'Total'\n  GROUP BY LocationDesc, YearStart\n),\n\n-- Poor nutrition rate per state-year (Total)\nnutrition_by_state_year AS (\n  SELECT LocationDesc, YearStart,\n         ROUND(AVG(Data_Value), 2) AS poor_nutrition_rate\n  FROM clean_base_dataset\n  WHERE Class = 'Fruits and Vegetables'\n    AND StratificationCategory1 = 'Total'\n  GROUP BY LocationDesc, YearStart\n),\n\n-- Income encoding: higher income bracket = higher number\nincome_encoded AS (\n  SELECT LocationDesc, YearStart,\n         AVG(CASE Stratification1\n           WHEN 'Less than $15,000' THEN 1\n           WHEN '$15,000 - $24,999' THEN 2\n           WHEN '$25,000 - $34,999' THEN 3\n           WHEN '$35,000 - $49,999' THEN 4\n           WHEN '$50,000 - $74,999' THEN 5\n           WHEN '$75,000 or greater' THEN 6\n           ELSE NULL END * Data_Value) / NULLIF(AVG(Data_Value), 0) AS weighted_income_index\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Income'\n  GROUP BY LocationDesc, YearStart\n),\n\n-- Education encoding: higher education = higher number\neducation_encoded AS (\n  SELECT LocationDesc, YearStart,\n         AVG(CASE Stratification1\n           WHEN 'Less than high school' THEN 1\n           WHEN 'High school graduate' THEN 2\n           WHEN 'Some college or technical school' THEN 3\n           WHEN 'College graduate' THEN 4\n           ELSE NULL END * Data_Value) / NULLIF(AVG(Data_Value), 0) AS weighted_education_index\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Education'\n  GROUP BY LocationDesc, YearStart\n),\n\n-- Age encoding: midpoint of each age bracket\nage_encoded AS (\n  SELECT LocationDesc, YearStart,\n         AVG(CASE Stratification1\n           WHEN '18 - 24' THEN 21\n           WHEN '25 - 34' THEN 30\n           WHEN '35 - 44' THEN 40\n           WHEN '45 - 54' THEN 50\n           WHEN '55 - 64' THEN 60\n           WHEN '65 or older' THEN 70\n           ELSE NULL END * Data_Value) / NULLIF(AVG(Data_Value), 0) AS weighted_age_index\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Age (years)'\n  GROUP BY LocationDesc, YearStart\n)\n\nSELECT\n  'Physical Inactivity' AS factor,\n  ROUND(CORR(o.obesity_rate, i.inactivity_rate), 4) AS correlation_coefficient,\n  COUNT(*) AS data_points,\n  CASE\n    WHEN ABS(CORR(o.obesity_rate, i.inactivity_rate)) >= 0.7 THEN 'Strong'\n    WHEN ABS(CORR(o.obesity_rate, i.inactivity_rate)) >= 0.4 THEN 'Moderate'\n    ELSE 'Weak'\n  END AS strength\nFROM obesity_by_state_year o\nJOIN inactivity_by_state_year i ON o.LocationDesc = i.LocationDesc AND o.YearStart = i.YearStart\n\nUNION ALL\n\nSELECT\n  'Poor Nutrition (Low Fruit/Veg)' AS factor,\n  ROUND(CORR(o.obesity_rate, n.poor_nutrition_rate), 4) AS correlation_coefficient,\n  COUNT(*) AS data_points,\n  CASE\n    WHEN ABS(CORR(o.obesity_rate, n.poor_nutrition_rate)) >= 0.7 THEN 'Strong'\n    WHEN ABS(CORR(o.obesity_rate, n.poor_nutrition_rate)) >= 0.4 THEN 'Moderate'\n    ELSE 'Weak'\n  END AS strength\nFROM obesity_by_state_year o\nJOIN nutrition_by_state_year n ON o.LocationDesc = n.LocationDesc AND o.YearStart = n.YearStart\n\nUNION ALL\n\nSELECT\n  'Lower Income (weighted index)' AS factor,\n  ROUND(CORR(o.obesity_rate, inc.weighted_income_index), 4) AS correlation_coefficient,\n  COUNT(*) AS data_points,\n  CASE\n    WHEN ABS(CORR(o.obesity_rate, inc.weighted_income_index)) >= 0.7 THEN 'Strong'\n    WHEN ABS(CORR(o.obesity_rate, inc.weighted_income_index)) >= 0.4 THEN 'Moderate'\n    ELSE 'Weak'\n  END AS strength\nFROM obesity_by_state_year o\nJOIN income_encoded inc ON o.LocationDesc = inc.LocationDesc AND o.YearStart = inc.YearStart\n\nUNION ALL\n\nSELECT\n  'Lower Education (weighted index)' AS factor,\n  ROUND(CORR(o.obesity_rate, edu.weighted_education_index), 4) AS correlation_coefficient,\n  COUNT(*) AS data_points,\n  CASE\n    WHEN ABS(CORR(o.obesity_rate, edu.weighted_education_index)) >= 0.7 THEN 'Strong'\n    WHEN ABS(CORR(o.obesity_rate, edu.weighted_education_index)) >= 0.4 THEN 'Moderate'\n    ELSE 'Weak'\n  END AS strength\nFROM obesity_by_state_year o\nJOIN education_encoded edu ON o.LocationDesc = edu.LocationDesc AND o.YearStart = edu.YearStart\n\nUNION ALL\n\nSELECT\n  'Older Age (weighted index)' AS factor,\n  ROUND(CORR(o.obesity_rate, age.weighted_age_index), 4) AS correlation_coefficient,\n  COUNT(*) AS data_points,\n  CASE\n    WHEN ABS(CORR(o.obesity_rate, age.weighted_age_index)) >= 0.7 THEN 'Strong'\n    WHEN ABS(CORR(o.obesity_rate, age.weighted_age_index)) >= 0.4 THEN 'Moderate'\n    ELSE 'Weak'\n  END AS strength\nFROM obesity_by_state_year o\nJOIN age_encoded age ON o.LocationDesc = age.LocationDesc AND o.YearStart = age.YearStart\n\nORDER BY ABS(correlation_coefficient) DESC\n"
}
inputs = {
    "data": [
        ctx["filter_null_data_values.filtered_data"]
    ],
    "data__sources": [
        {
            "node": "filter_null_data_values",
            "output_port": "filtered_data",
            "name": "clean_base_dataset",
            "df_name": "clean_base_dataset"
        }
    ]
}
out = run(config, inputs, spark)
ctx["correlation_analysis.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["correlation_analysis.result"])

In [0]:
"""
id: q1_state_obesity_trends
template: sql
templateVersion: 1.0.0
name: Q1 - State Obesity Trends Over Time
position:
  x: 1359
  y: 618
previewCodeHash: 06f5f40fdd88ef33
previewMode: full
config:
  query: |
    WITH national AS (
      SELECT YearStart, ROUND(AVG(Data_Value), 2) AS national_avg
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Total'
      GROUP BY YearStart
    )
    SELECT
      s.LocationDesc AS state,
      s.YearStart AS year,
      ROUND(AVG(s.Data_Value), 2) AS obesity_rate,
      n.national_avg,
      ROUND(AVG(s.Data_Value) - n.national_avg, 2) AS vs_national_avg
    FROM clean_base_dataset s
    JOIN national n ON s.YearStart = n.YearStart
    WHERE s.Class = 'Obesity / Weight Status'
      AND s.StratificationCategory1 = 'Total'
    GROUP BY s.LocationDesc, s.YearStart, n.national_avg
    ORDER BY s.YearStart DESC, obesity_rate DESC
input:
  - node: filter_null_data_values
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "WITH national AS (\n  SELECT YearStart, ROUND(AVG(Data_Value), 2) AS national_avg\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Total'\n  GROUP BY YearStart\n)\nSELECT\n  s.LocationDesc AS state,\n  s.YearStart AS year,\n  ROUND(AVG(s.Data_Value), 2) AS obesity_rate,\n  n.national_avg,\n  ROUND(AVG(s.Data_Value) - n.national_avg, 2) AS vs_national_avg\nFROM clean_base_dataset s\nJOIN national n ON s.YearStart = n.YearStart\nWHERE s.Class = 'Obesity / Weight Status'\n  AND s.StratificationCategory1 = 'Total'\nGROUP BY s.LocationDesc, s.YearStart, n.national_avg\nORDER BY s.YearStart DESC, obesity_rate DESC\n"
}
inputs = {
    "data": [
        ctx["filter_null_data_values.filtered_data"]
    ],
    "data__sources": [
        {
            "node": "filter_null_data_values",
            "output_port": "filtered_data",
            "name": "clean_base_dataset",
            "df_name": "clean_base_dataset"
        }
    ]
}
out = run(config, inputs, spark)
ctx["q1_state_obesity_trends.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["q1_state_obesity_trends.result"])

In [0]:
"""
id: q1a_obesity_by_sex_per_state
template: sql
templateVersion: 1.0.0
name: Q1a - Obesity by Sex per State
position:
  x: 1640
  y: 618
description:
  text: Calculate obesity rates by sex for each state and compare to the national average.
  hash: 4702f661
previewCodeHash: 13cfd3c82f42e60d
previewMode: full
config:
  query: |
    WITH state_total AS (
      SELECT
        LocationDesc,
        ROUND(AVG(Data_Value), 2) AS state_overall_rate
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Total'
      GROUP BY LocationDesc
    ),
    national AS (
      SELECT ROUND(AVG(Data_Value), 2) AS national_avg
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Total'
    )
    SELECT
      s.LocationDesc AS state,
      t.state_overall_rate,
      s.Stratification1 AS sex,
      ROUND(AVG(s.Data_Value), 2) AS obesity_rate,
      n.national_avg,
      ROUND(AVG(s.Data_Value) - n.national_avg, 2) AS vs_national_avg
    FROM clean_base_dataset s
    JOIN state_total t ON s.LocationDesc = t.LocationDesc
    CROSS JOIN national n
    WHERE s.Class = 'Obesity / Weight Status'
      AND s.StratificationCategory1 = 'Sex'
    GROUP BY s.LocationDesc, s.Stratification1, t.state_overall_rate, n.national_avg
    ORDER BY t.state_overall_rate DESC, s.LocationDesc, sex
input:
  - node: filter_null_data_values
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "WITH state_total AS (\n  SELECT\n    LocationDesc,\n    ROUND(AVG(Data_Value), 2) AS state_overall_rate\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Total'\n  GROUP BY LocationDesc\n),\nnational AS (\n  SELECT ROUND(AVG(Data_Value), 2) AS national_avg\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Total'\n)\nSELECT\n  s.LocationDesc AS state,\n  t.state_overall_rate,\n  s.Stratification1 AS sex,\n  ROUND(AVG(s.Data_Value), 2) AS obesity_rate,\n  n.national_avg,\n  ROUND(AVG(s.Data_Value) - n.national_avg, 2) AS vs_national_avg\nFROM clean_base_dataset s\nJOIN state_total t ON s.LocationDesc = t.LocationDesc\nCROSS JOIN national n\nWHERE s.Class = 'Obesity / Weight Status'\n  AND s.StratificationCategory1 = 'Sex'\nGROUP BY s.LocationDesc, s.Stratification1, t.state_overall_rate, n.national_avg\nORDER BY t.state_overall_rate DESC, s.LocationDesc, sex\n"
}
inputs = {
    "data": [
        ctx["filter_null_data_values.filtered_data"]
    ],
    "data__sources": [
        {
            "node": "filter_null_data_values",
            "output_port": "filtered_data",
            "name": "clean_base_dataset",
            "df_name": "clean_base_dataset"
        }
    ]
}
out = run(config, inputs, spark)
ctx["q1a_obesity_by_sex_per_state.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["q1a_obesity_by_sex_per_state.result"])

In [0]:
"""
id: q1b_obesity_by_age_per_state
template: sql
templateVersion: 1.0.0
name: Q1b - Obesity by Age per State
position:
  x: 1640
  y: 778
description:
  text: Calculate average obesity rates by state and age group, compare with national average, and order states by overall obesity rate.
  hash: 1948dae5
previewCodeHash: 9c0968df0db493cc
previewMode: full
config:
  query: |
    WITH state_total AS (
      SELECT
        LocationDesc,
        ROUND(AVG(Data_Value), 2) AS state_overall_rate
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Total'
      GROUP BY LocationDesc
    ),
    national AS (
      SELECT ROUND(AVG(Data_Value), 2) AS national_avg
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Total'
    )
    SELECT
      s.LocationDesc AS state,
      t.state_overall_rate,
      s.Stratification1 AS age_group,
      ROUND(AVG(s.Data_Value), 2) AS obesity_rate,
      n.national_avg,
      ROUND(AVG(s.Data_Value) - n.national_avg, 2) AS vs_national_avg
    FROM clean_base_dataset s
    JOIN state_total t ON s.LocationDesc = t.LocationDesc
    CROSS JOIN national n
    WHERE s.Class = 'Obesity / Weight Status'
      AND s.StratificationCategory1 = 'Age (years)'
    GROUP BY s.LocationDesc, s.Stratification1, t.state_overall_rate, n.national_avg
    ORDER BY t.state_overall_rate DESC, s.LocationDesc, age_group
input:
  - node: filter_null_data_values
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "WITH state_total AS (\n  SELECT\n    LocationDesc,\n    ROUND(AVG(Data_Value), 2) AS state_overall_rate\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Total'\n  GROUP BY LocationDesc\n),\nnational AS (\n  SELECT ROUND(AVG(Data_Value), 2) AS national_avg\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Total'\n)\nSELECT\n  s.LocationDesc AS state,\n  t.state_overall_rate,\n  s.Stratification1 AS age_group,\n  ROUND(AVG(s.Data_Value), 2) AS obesity_rate,\n  n.national_avg,\n  ROUND(AVG(s.Data_Value) - n.national_avg, 2) AS vs_national_avg\nFROM clean_base_dataset s\nJOIN state_total t ON s.LocationDesc = t.LocationDesc\nCROSS JOIN national n\nWHERE s.Class = 'Obesity / Weight Status'\n  AND s.StratificationCategory1 = 'Age (years)'\nGROUP BY s.LocationDesc, s.Stratification1, t.state_overall_rate, n.national_avg\nORDER BY t.state_overall_rate DESC, s.LocationDesc, age_group\n"
}
inputs = {
    "data": [
        ctx["filter_null_data_values.filtered_data"]
    ],
    "data__sources": [
        {
            "node": "filter_null_data_values",
            "output_port": "filtered_data",
            "name": "clean_base_dataset",
            "df_name": "clean_base_dataset"
        }
    ]
}
out = run(config, inputs, spark)
ctx["q1b_obesity_by_age_per_state.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["q1b_obesity_by_age_per_state.result"])

In [0]:
"""
id: q1c_obesity_by_education_per_state
template: sql
templateVersion: 1.0.0
name: Q1c - Obesity by Education per State
position:
  x: 1640
  y: 938
description:
  text: Calculate obesity rates by education level for each state compared to the state and national averages.
  hash: 4f00ce4e
previewCodeHash: ecf7fcb6f72b7e78
previewMode: full
config:
  query: |
    WITH state_total AS (
      SELECT
        LocationDesc,
        ROUND(AVG(Data_Value), 2) AS state_overall_rate
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Total'
      GROUP BY LocationDesc
    ),
    national AS (
      SELECT ROUND(AVG(Data_Value), 2) AS national_avg
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Total'
    )
    SELECT
      s.LocationDesc AS state,
      t.state_overall_rate,
      s.Stratification1 AS education_level,
      ROUND(AVG(s.Data_Value), 2) AS obesity_rate,
      n.national_avg,
      ROUND(AVG(s.Data_Value) - n.national_avg, 2) AS vs_national_avg
    FROM clean_base_dataset s
    JOIN state_total t ON s.LocationDesc = t.LocationDesc
    CROSS JOIN national n
    WHERE s.Class = 'Obesity / Weight Status'
      AND s.StratificationCategory1 = 'Education'
    GROUP BY s.LocationDesc, s.Stratification1, t.state_overall_rate, n.national_avg
    ORDER BY t.state_overall_rate DESC, s.LocationDesc, education_level
input:
  - node: filter_null_data_values
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "WITH state_total AS (\n  SELECT\n    LocationDesc,\n    ROUND(AVG(Data_Value), 2) AS state_overall_rate\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Total'\n  GROUP BY LocationDesc\n),\nnational AS (\n  SELECT ROUND(AVG(Data_Value), 2) AS national_avg\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Total'\n)\nSELECT\n  s.LocationDesc AS state,\n  t.state_overall_rate,\n  s.Stratification1 AS education_level,\n  ROUND(AVG(s.Data_Value), 2) AS obesity_rate,\n  n.national_avg,\n  ROUND(AVG(s.Data_Value) - n.national_avg, 2) AS vs_national_avg\nFROM clean_base_dataset s\nJOIN state_total t ON s.LocationDesc = t.LocationDesc\nCROSS JOIN national n\nWHERE s.Class = 'Obesity / Weight Status'\n  AND s.StratificationCategory1 = 'Education'\nGROUP BY s.LocationDesc, s.Stratification1, t.state_overall_rate, n.national_avg\nORDER BY t.state_overall_rate DESC, s.LocationDesc, education_level\n"
}
inputs = {
    "data": [
        ctx["filter_null_data_values.filtered_data"]
    ],
    "data__sources": [
        {
            "node": "filter_null_data_values",
            "output_port": "filtered_data",
            "name": "clean_base_dataset",
            "df_name": "clean_base_dataset"
        }
    ]
}
out = run(config, inputs, spark)
ctx["q1c_obesity_by_education_per_state.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["q1c_obesity_by_education_per_state.result"])

In [0]:
"""
id: q1d_obesity_by_income_per_state
template: sql
templateVersion: 1.0.0
name: Q1d - Obesity by Income per State
position:
  x: 1640
  y: 1098
description:
  text: Calculate average obesity rates by income bracket in each state, compare to national average, and order states by overall obesity rate.
  hash: 0bc74dad
previewCodeHash: 8dd9491c651ecf5c
previewMode: full
config:
  query: |
    WITH state_total AS (
      SELECT
        LocationDesc,
        ROUND(AVG(Data_Value), 2) AS state_overall_rate
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Total'
      GROUP BY LocationDesc
    ),
    national AS (
      SELECT ROUND(AVG(Data_Value), 2) AS national_avg
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Total'
    )
    SELECT
      s.LocationDesc AS state,
      t.state_overall_rate,
      s.Stratification1 AS income_bracket,
      ROUND(AVG(s.Data_Value), 2) AS obesity_rate,
      n.national_avg,
      ROUND(AVG(s.Data_Value) - n.national_avg, 2) AS vs_national_avg
    FROM clean_base_dataset s
    JOIN state_total t ON s.LocationDesc = t.LocationDesc
    CROSS JOIN national n
    WHERE s.Class = 'Obesity / Weight Status'
      AND s.StratificationCategory1 = 'Income'
    GROUP BY s.LocationDesc, s.Stratification1, t.state_overall_rate, n.national_avg
    ORDER BY t.state_overall_rate DESC, s.LocationDesc, income_bracket
input:
  - node: filter_null_data_values
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "WITH state_total AS (\n  SELECT\n    LocationDesc,\n    ROUND(AVG(Data_Value), 2) AS state_overall_rate\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Total'\n  GROUP BY LocationDesc\n),\nnational AS (\n  SELECT ROUND(AVG(Data_Value), 2) AS national_avg\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Total'\n)\nSELECT\n  s.LocationDesc AS state,\n  t.state_overall_rate,\n  s.Stratification1 AS income_bracket,\n  ROUND(AVG(s.Data_Value), 2) AS obesity_rate,\n  n.national_avg,\n  ROUND(AVG(s.Data_Value) - n.national_avg, 2) AS vs_national_avg\nFROM clean_base_dataset s\nJOIN state_total t ON s.LocationDesc = t.LocationDesc\nCROSS JOIN national n\nWHERE s.Class = 'Obesity / Weight Status'\n  AND s.StratificationCategory1 = 'Income'\nGROUP BY s.LocationDesc, s.Stratification1, t.state_overall_rate, n.national_avg\nORDER BY t.state_overall_rate DESC, s.LocationDesc, income_bracket\n"
}
inputs = {
    "data": [
        ctx["filter_null_data_values.filtered_data"]
    ],
    "data__sources": [
        {
            "node": "filter_null_data_values",
            "output_port": "filtered_data",
            "name": "clean_base_dataset",
            "df_name": "clean_base_dataset"
        }
    ]
}
out = run(config, inputs, spark)
ctx["q1d_obesity_by_income_per_state.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["q1d_obesity_by_income_per_state.result"])

In [0]:
"""
id: q1e_obesity_by_race_per_state
template: sql
templateVersion: 1.0.0
name: Q1e - Obesity by Race/Ethnicity per State
position:
  x: 1640
  y: 1258
description:
  text: Calculate and compare average obesity rates by race and state to the national average.
  hash: 4163582b
previewCodeHash: 3768b46f43a1181b
previewMode: full
config:
  query: |
    WITH state_total AS (
      SELECT
        LocationDesc,
        ROUND(AVG(Data_Value), 2) AS state_overall_rate
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Total'
      GROUP BY LocationDesc
    ),
    national AS (
      SELECT ROUND(AVG(Data_Value), 2) AS national_avg
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Total'
    )
    SELECT
      s.LocationDesc AS state,
      t.state_overall_rate,
      s.Stratification1 AS race_ethnicity,
      ROUND(AVG(s.Data_Value), 2) AS obesity_rate,
      n.national_avg,
      ROUND(AVG(s.Data_Value) - n.national_avg, 2) AS vs_national_avg
    FROM clean_base_dataset s
    JOIN state_total t ON s.LocationDesc = t.LocationDesc
    CROSS JOIN national n
    WHERE s.Class = 'Obesity / Weight Status'
      AND s.StratificationCategory1 = 'Race/Ethnicity'
    GROUP BY s.LocationDesc, s.Stratification1, t.state_overall_rate, n.national_avg
    ORDER BY t.state_overall_rate DESC, s.LocationDesc, race_ethnicity
input:
  - node: filter_null_data_values
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "WITH state_total AS (\n  SELECT\n    LocationDesc,\n    ROUND(AVG(Data_Value), 2) AS state_overall_rate\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Total'\n  GROUP BY LocationDesc\n),\nnational AS (\n  SELECT ROUND(AVG(Data_Value), 2) AS national_avg\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Total'\n)\nSELECT\n  s.LocationDesc AS state,\n  t.state_overall_rate,\n  s.Stratification1 AS race_ethnicity,\n  ROUND(AVG(s.Data_Value), 2) AS obesity_rate,\n  n.national_avg,\n  ROUND(AVG(s.Data_Value) - n.national_avg, 2) AS vs_national_avg\nFROM clean_base_dataset s\nJOIN state_total t ON s.LocationDesc = t.LocationDesc\nCROSS JOIN national n\nWHERE s.Class = 'Obesity / Weight Status'\n  AND s.StratificationCategory1 = 'Race/Ethnicity'\nGROUP BY s.LocationDesc, s.Stratification1, t.state_overall_rate, n.national_avg\nORDER BY t.state_overall_rate DESC, s.LocationDesc, race_ethnicity\n"
}
inputs = {
    "data": [
        ctx["filter_null_data_values.filtered_data"]
    ],
    "data__sources": [
        {
            "node": "filter_null_data_values",
            "output_port": "filtered_data",
            "name": "clean_base_dataset",
            "df_name": "clean_base_dataset"
        }
    ]
}
out = run(config, inputs, spark)
ctx["q1e_obesity_by_race_per_state.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["q1e_obesity_by_race_per_state.result"])

In [0]:
"""
id: q2_obesity_by_income
template: sql
templateVersion: 1.0.0
name: Q2 - Obesity by Income Level
position:
  x: 1372
  y: 736
previewCodeHash: b2217aa9800d31c3
previewMode: full
config:
  query: |
    WITH national AS (
      SELECT ROUND(AVG(Data_Value), 2) AS national_avg
      FROM clean_base_dataset
      WHERE Class = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Total'
    )
    SELECT
      Stratification1 AS income_bracket,
      ROUND(AVG(Data_Value), 2) AS avg_obesity_rate,
      national.national_avg,
      ROUND(AVG(Data_Value) - national.national_avg, 2) AS vs_national_avg,
      COUNT(*) AS data_points
    FROM clean_base_dataset
    CROSS JOIN national
    WHERE Class = 'Obesity / Weight Status'
      AND StratificationCategory1 = 'Income'
    GROUP BY Stratification1, national.national_avg
    ORDER BY avg_obesity_rate DESC
input:
  - node: filter_null_data_values
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "WITH national AS (\n  SELECT ROUND(AVG(Data_Value), 2) AS national_avg\n  FROM clean_base_dataset\n  WHERE Class = 'Obesity / Weight Status'\n    AND StratificationCategory1 = 'Total'\n)\nSELECT\n  Stratification1 AS income_bracket,\n  ROUND(AVG(Data_Value), 2) AS avg_obesity_rate,\n  national.national_avg,\n  ROUND(AVG(Data_Value) - national.national_avg, 2) AS vs_national_avg,\n  COUNT(*) AS data_points\nFROM clean_base_dataset\nCROSS JOIN national\nWHERE Class = 'Obesity / Weight Status'\n  AND StratificationCategory1 = 'Income'\nGROUP BY Stratification1, national.national_avg\nORDER BY avg_obesity_rate DESC\n"
}
inputs = {
    "data": [
        ctx["filter_null_data_values.filtered_data"]
    ],
    "data__sources": [
        {
            "node": "filter_null_data_values",
            "output_port": "filtered_data",
            "name": "clean_base_dataset",
            "df_name": "clean_base_dataset"
        }
    ]
}
out = run(config, inputs, spark)
ctx["q2_obesity_by_income.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["q2_obesity_by_income.result"])

In [0]:
"""
id: q3_physical_activity_by_sex
template: sql
templateVersion: 1.0.0
name: Q3 - Physical Activity by Sex
position:
  x: 1377.3973917804356
  y: 882.4838293340958
previewCodeHash: 4b2bbdae8e524b15
previewMode: full
config:
  query: |
    WITH national AS (
      SELECT ROUND(AVG(Data_Value), 2) AS national_avg
      FROM clean_base_dataset
      WHERE Class = 'Physical Activity'
        AND StratificationCategory1 = 'Total'
    )
    SELECT
      Stratification1 AS sex,
      ROUND(AVG(Data_Value), 2) AS avg_inactivity_rate,
      national.national_avg,
      ROUND(AVG(Data_Value) - national.national_avg, 2) AS vs_national_avg,
      COUNT(*) AS data_points
    FROM clean_base_dataset
    CROSS JOIN national
    WHERE Class = 'Physical Activity'
      AND StratificationCategory1 = 'Sex'
    GROUP BY Stratification1, national.national_avg
    ORDER BY avg_inactivity_rate DESC
input:
  - node: filter_null_data_values
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "WITH national AS (\n  SELECT ROUND(AVG(Data_Value), 2) AS national_avg\n  FROM clean_base_dataset\n  WHERE Class = 'Physical Activity'\n    AND StratificationCategory1 = 'Total'\n)\nSELECT\n  Stratification1 AS sex,\n  ROUND(AVG(Data_Value), 2) AS avg_inactivity_rate,\n  national.national_avg,\n  ROUND(AVG(Data_Value) - national.national_avg, 2) AS vs_national_avg,\n  COUNT(*) AS data_points\nFROM clean_base_dataset\nCROSS JOIN national\nWHERE Class = 'Physical Activity'\n  AND StratificationCategory1 = 'Sex'\nGROUP BY Stratification1, national.national_avg\nORDER BY avg_inactivity_rate DESC\n"
}
inputs = {
    "data": [
        ctx["filter_null_data_values.filtered_data"]
    ],
    "data__sources": [
        {
            "node": "filter_null_data_values",
            "output_port": "filtered_data",
            "name": "clean_base_dataset",
            "df_name": "clean_base_dataset"
        }
    ]
}
out = run(config, inputs, spark)
ctx["q3_physical_activity_by_sex.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["q3_physical_activity_by_sex.result"])

In [0]:
"""
id: q4_nutrition_by_age
template: sql
templateVersion: 1.0.0
name: Q4 - Nutrition Risk by Age Group
position:
  x: 1368.1600086391725
  y: 1021.604369091662
previewCodeHash: aaa175d5234e47c4
previewMode: full
config:
  query: |
    WITH national AS (
      SELECT ROUND(AVG(Data_Value), 2) AS national_avg
      FROM clean_base_dataset
      WHERE Class = 'Fruits and Vegetables'
        AND StratificationCategory1 = 'Total'
    )
    SELECT
      Stratification1 AS age_group,
      ROUND(AVG(Data_Value), 2) AS avg_rate,
      national.national_avg,
      ROUND(AVG(Data_Value) - national.national_avg, 2) AS vs_national_avg,
      COUNT(*) AS data_points
    FROM clean_base_dataset
    CROSS JOIN national
    WHERE Class = 'Fruits and Vegetables'
      AND StratificationCategory1 = 'Age (years)'
    GROUP BY Stratification1, national.national_avg
    ORDER BY avg_rate ASC
input:
  - node: filter_null_data_values
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "WITH national AS (\n  SELECT ROUND(AVG(Data_Value), 2) AS national_avg\n  FROM clean_base_dataset\n  WHERE Class = 'Fruits and Vegetables'\n    AND StratificationCategory1 = 'Total'\n)\nSELECT\n  Stratification1 AS age_group,\n  ROUND(AVG(Data_Value), 2) AS avg_rate,\n  national.national_avg,\n  ROUND(AVG(Data_Value) - national.national_avg, 2) AS vs_national_avg,\n  COUNT(*) AS data_points\nFROM clean_base_dataset\nCROSS JOIN national\nWHERE Class = 'Fruits and Vegetables'\n  AND StratificationCategory1 = 'Age (years)'\nGROUP BY Stratification1, national.national_avg\nORDER BY avg_rate ASC\n"
}
inputs = {
    "data": [
        ctx["filter_null_data_values.filtered_data"]
    ],
    "data__sources": [
        {
            "node": "filter_null_data_values",
            "output_port": "filtered_data",
            "name": "clean_base_dataset",
            "df_name": "clean_base_dataset"
        }
    ]
}
out = run(config, inputs, spark)
ctx["q4_nutrition_by_age.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["q4_nutrition_by_age.result"])

In [0]:
"""
id: q5_education_vs_obesity
template: sql
templateVersion: 1.0.0
name: Q5 - Education vs Obesity & Inactivity
position:
  x: 1381.2755704341303
  y: 1205.817713943892
previewCodeHash: f5590e6de447efae
previewMode: full
config:
  query: |
    WITH national AS (
      SELECT Class, ROUND(AVG(Data_Value), 2) AS national_avg
      FROM clean_base_dataset
      WHERE StratificationCategory1 = 'Total'
        AND Class IN ('Obesity / Weight Status', 'Physical Activity')
      GROUP BY Class
    )
    SELECT
      Stratification1 AS education_level,
      e.Class,
      ROUND(AVG(e.Data_Value), 2) AS avg_rate,
      n.national_avg,
      ROUND(AVG(e.Data_Value) - n.national_avg, 2) AS vs_national_avg,
      COUNT(*) AS data_points
    FROM clean_base_dataset e
    JOIN national n ON e.Class = n.Class
    WHERE e.StratificationCategory1 = 'Education'
      AND e.Class IN ('Obesity / Weight Status', 'Physical Activity')
    GROUP BY Stratification1, e.Class, n.national_avg
    ORDER BY e.Class, avg_rate DESC
input:
  - node: filter_null_data_values
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "WITH national AS (\n  SELECT Class, ROUND(AVG(Data_Value), 2) AS national_avg\n  FROM clean_base_dataset\n  WHERE StratificationCategory1 = 'Total'\n    AND Class IN ('Obesity / Weight Status', 'Physical Activity')\n  GROUP BY Class\n)\nSELECT\n  Stratification1 AS education_level,\n  e.Class,\n  ROUND(AVG(e.Data_Value), 2) AS avg_rate,\n  n.national_avg,\n  ROUND(AVG(e.Data_Value) - n.national_avg, 2) AS vs_national_avg,\n  COUNT(*) AS data_points\nFROM clean_base_dataset e\nJOIN national n ON e.Class = n.Class\nWHERE e.StratificationCategory1 = 'Education'\n  AND e.Class IN ('Obesity / Weight Status', 'Physical Activity')\nGROUP BY Stratification1, e.Class, n.national_avg\nORDER BY e.Class, avg_rate DESC\n"
}
inputs = {
    "data": [
        ctx["filter_null_data_values.filtered_data"]
    ],
    "data__sources": [
        {
            "node": "filter_null_data_values",
            "output_port": "filtered_data",
            "name": "clean_base_dataset",
            "df_name": "clean_base_dataset"
        }
    ]
}
out = run(config, inputs, spark)
ctx["q5_education_vs_obesity.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["q5_education_vs_obesity.result"])

In [0]:
"""
id: q6_racial_health_disparities
template: sql
templateVersion: 1.0.0
name: Q6 - Racial & Ethnic Health Disparities
position:
  x: 1365.5452487587045
  y: 1376.9251859628246
previewCodeHash: acb77e7229bf665a
previewMode: full
config:
  query: |
    WITH national AS (
      SELECT Class, ROUND(AVG(Data_Value), 2) AS national_avg
      FROM clean_base_dataset
      WHERE StratificationCategory1 = 'Total'
        AND Class IN ('Obesity / Weight Status', 'Physical Activity')
      GROUP BY Class
    )
    SELECT
      Stratification1 AS race_ethnicity,
      r.Class,
      ROUND(AVG(r.Data_Value), 2) AS avg_rate,
      n.national_avg,
      ROUND(AVG(r.Data_Value) - n.national_avg, 2) AS vs_national_avg,
      COUNT(*) AS data_points
    FROM clean_base_dataset r
    JOIN national n ON r.Class = n.Class
    WHERE r.StratificationCategory1 = 'Race/Ethnicity'
      AND r.Class IN ('Obesity / Weight Status', 'Physical Activity')
    GROUP BY Stratification1, r.Class, n.national_avg
    ORDER BY r.Class, avg_rate DESC
input:
  - node: filter_null_data_values
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "query": "WITH national AS (\n  SELECT Class, ROUND(AVG(Data_Value), 2) AS national_avg\n  FROM clean_base_dataset\n  WHERE StratificationCategory1 = 'Total'\n    AND Class IN ('Obesity / Weight Status', 'Physical Activity')\n  GROUP BY Class\n)\nSELECT\n  Stratification1 AS race_ethnicity,\n  r.Class,\n  ROUND(AVG(r.Data_Value), 2) AS avg_rate,\n  n.national_avg,\n  ROUND(AVG(r.Data_Value) - n.national_avg, 2) AS vs_national_avg,\n  COUNT(*) AS data_points\nFROM clean_base_dataset r\nJOIN national n ON r.Class = n.Class\nWHERE r.StratificationCategory1 = 'Race/Ethnicity'\n  AND r.Class IN ('Obesity / Weight Status', 'Physical Activity')\nGROUP BY Stratification1, r.Class, n.national_avg\nORDER BY r.Class, avg_rate DESC\n"
}
inputs = {
    "data": [
        ctx["filter_null_data_values.filtered_data"]
    ],
    "data__sources": [
        {
            "node": "filter_null_data_values",
            "output_port": "filtered_data",
            "name": "clean_base_dataset",
            "df_name": "clean_base_dataset"
        }
    ]
}
out = run(config, inputs, spark)
ctx["q6_racial_health_disparities.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["q6_racial_health_disparities.result"])